# Getting the Silver Data and Pushing to Gold

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
df = spark.read.table("ecom.silver.fact_silver_payments")
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
table_name = "ecom.gold.fact_gold_payments"
silver_df = df.select(
    "order_id",
    "payment_sequential",
    "payment_type",
    "payment_installments",
    "payment_value",
    "ingest_date",
    "file_date",
    "created_timestamp",
    "updated_timestamp",
    "data_quality_flag"
).dropDuplicates(["order_id", "payment_sequential"])
 

In [0]:

if spark.catalog.tableExists(table_name):

    gold_delta = DeltaTable.forName(spark, table_name)

    (
        gold_delta.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.order_id = source.order_id
            AND target.payment_sequential = source.payment_sequential
            """
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

else:
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )